# 04 · Supervised Learning: Classification — deep dive

> **Course:** Machine Learning Paradigms (DS602) · IIIT Dharwad · companion to [`../04-Supervised-Learning-Classification.md`](../04-Supervised-Learning-Classification.md)
>
> Every number quoted in the note is produced here. Run top to bottom (NumPy, scikit-learn ≥ 1.2; no downloads needed).

| Part | Topic |
|:-:|:--|
| A | Linear regression for classification (slides 6–11) and why it breaks |
| B | The sigmoid (slides 12–13) |
| C | Logistic regression on the word-count data (slides 14–17): separable data, the slide's weights |
| D | Decision boundaries in 2-D: linear and circular (slides 18–19) |
| E | Why not MSE: non-convex vs convex cost (slides 20–25) |
| F | Logistic regression by gradient descent from scratch |
| G | Metrics: LR vs DT, terrorist, spam, F1 (slides 26–40) |
| H | Thresholds, ROC and AUC (slide 26 lists AUC) |
| I | Multi-class: one-vs-rest, softmax, averaging (iris) |
| J | Model comparison demo: six classifiers, cross-validation (breast cancer) |
| K | Class imbalance: class weights and threshold moving |
| L | Practice-problem solutions |

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.datasets import load_iris, load_breast_cancer, make_classification
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score, recall_score, f1_score,
                             classification_report, roc_auc_score, roc_curve, log_loss, balanced_accuracy_score)

np.set_printoptions(precision=5, suppress=True)
print("numpy", np.__version__, "| scikit-learn", sklearn.__version__)

numpy 2.4.6 | scikit-learn 1.9.0


## Part A — Linear regression for classification (slides 6–11)

In [2]:
X = np.array([27, 2, 100, 40, 14], dtype=float)
y = np.array([0, 0, 1, 1, 0])
lin = LinearRegression().fit(X[:, None], y)
w, b = lin.coef_[0], lin.intercept_
print(f"OLS line: y_hat = {w:.5f} x + ({b:.5f})")
for xi, yi in zip(X, y):
    print(f"x = {xi:5.0f}  y = {yi}  y_hat = {w * xi + b:.5f}  class = {int(w * xi + b >= 0.5)}")
print(f"x = 20 (unknown): y_hat = {w * 20 + b:.5f} -> class {int(w * 20 + b >= 0.5)}")
print(f"Threshold 0.5 crossed at x = (0.5 - b) / w = {(0.5 - b) / w:.2f}")

OLS line: y_hat = 0.01146 x + (-0.01928)
x =    27  y = 0  y_hat = 0.29003  class = 0
x =     2  y = 0  y_hat = 0.00364  class = 0
x =   100  y = 1  y_hat = 1.12629  class = 1
x =    40  y = 1  y_hat = 0.43895  class = 0
x =    14  y = 0  y_hat = 0.14110  class = 0
x = 20 (unknown): y_hat = 0.20984 -> class 0
Threshold 0.5 crossed at x = (0.5 - b) / w = 45.33


Slide 8–10: with threshold 0.5 the line classifies x = 40 (a real 1) as 0. Add one very long positive review and the line tilts further:

In [3]:
X2 = np.append(X, 400); y2 = np.append(y, 1)
lin2 = LinearRegression().fit(X2[:, None], y2)
w2, b2 = lin2.coef_[0], lin2.intercept_
print(f"With x = 400 added: y_hat = {w2:.5f} x + ({b2:.5f}); 0.5 crossed at x = {(0.5 - b2) / w2:.2f}")
print("Predictions:", np.round(w2 * X2 + b2, 4), "-> classes", (w2 * X2 + b2 >= 0.5).astype(int))
print("Logistic boundary without / with the extra point:",
      round(-LogisticRegression(C=1e4, max_iter=10000).fit(X[:, None], y).intercept_[0] /
            LogisticRegression(C=1e4, max_iter=10000).fit(X[:, None], y).coef_[0, 0], 2), "/",
      round(-LogisticRegression(C=1e4, max_iter=10000).fit(X2[:, None], y2).intercept_[0] /
            LogisticRegression(C=1e4, max_iter=10000).fit(X2[:, None], y2).coef_[0, 0], 2))

With x = 400 added: y_hat = 0.00214 x + (0.29163); 0.5 crossed at x = 97.17
Predictions: [0.3495 0.2959 0.5061 0.3774 0.3217 1.1494] -> classes [0 0 1 0 0 1]
Logistic boundary without / with the extra point: 33.45 / 33.32


## Part B — The sigmoid

In [4]:
sig = lambda z: 1 / (1 + np.exp(-z))
for z in [-10, -4, -2, -1, 0, 1, 2, 4, 10]:
    print(f"g({z:3d}) = {sig(z):.5f}")
print("\nsymmetry g(-z) = 1 - g(z):", np.isclose(sig(-1.7), 1 - sig(1.7)))
print("derivative g'(0) = g(0)(1-g(0)) =", sig(0) * (1 - sig(0)))
print("logit (inverse): log(p/(1-p)) at p = 0.8 ->", round(np.log(0.8 / 0.2), 5), "| g of that ->", round(sig(np.log(4)), 5))

g(-10) = 0.00005
g( -4) = 0.01799
g( -2) = 0.11920
g( -1) = 0.26894
g(  0) = 0.50000
g(  1) = 0.73106
g(  2) = 0.88080
g(  4) = 0.98201
g( 10) = 0.99995

symmetry g(-z) = 1 - g(z): True
derivative g'(0) = g(0)(1-g(0)) = 0.25
logit (inverse): log(p/(1-p)) at p = 0.8 -> 1.38629 | g of that -> 0.8


## Part C — Logistic regression on the word-count data (slides 14–17)

In [5]:
xw = np.array([2, 3, 5, 26, 27, 28, 29, 30, 1, 4, 6, 31, 32], dtype=float)
yw = np.array([0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1])
w_s, b_s = 1.9292, -30.8841          # slide 17
print("Slide boundary x =", round(-b_s / w_s, 4))
for x in [6, 15, 16, 17, 20, 26]:
    z = w_s * x + b_s
    print(f"x = {x:2d}: z = {z:8.4f}  P(y=1) = {sig(z):.4f}")
print("\nLargest 0 =", xw[yw == 0].max(), "| smallest 1 =", xw[yw == 1].min(), "-> perfectly separable; midpoint", (6 + 26) / 2)

Slide boundary x = 16.0088
x =  6: z = -19.3089  P(y=1) = 0.0000
x = 15: z =  -1.9461  P(y=1) = 0.1250
x = 16: z =  -0.0169  P(y=1) = 0.4958
x = 17: z =   1.9123  P(y=1) = 0.8713
x = 20: z =   7.6999  P(y=1) = 0.9995
x = 26: z =  19.2751  P(y=1) = 1.0000

Largest 0 = 6.0 | smallest 1 = 26.0 -> perfectly separable; midpoint 16.0


In [6]:
# Separable data: the likelihood keeps improving as the weights grow, so the fitted weights depend on regularisation (C)
rows = []
for C in [0.1, 1, 10, 100, 1e4, 1e6]:
    m = LogisticRegression(C=C, max_iter=100000).fit(xw[:, None], yw)
    p = m.predict_proba(xw[:, None])[:, 1]
    rows.append({"C": C, "w": m.coef_[0, 0], "b": m.intercept_[0], "boundary": -m.intercept_[0] / m.coef_[0, 0],
                 "P(17)": m.predict_proba([[17]])[0, 1], "train log-loss": log_loss(yw, p), "train acc": m.score(xw[:, None], yw)})
print(pd.DataFrame(rows).round(4).to_string(index=False))
z = w_s * xw + b_s
print("\nSlide weights: train log-loss =", round(log_loss(yw, sig(z)), 6))

        C      w        b  boundary  P(17)  train log-loss  train acc
      0.1 0.3150  -5.0160   15.9218 0.5841          0.0209        1.0
      1.0 0.4761  -7.5937   15.9489 0.6226          0.0033        1.0
     10.0 0.6532 -10.4451   15.9901 0.6592          0.0005        1.0
    100.0 0.8366 -13.3436   15.9499 0.7065          0.0001        1.0
  10000.0 1.0192 -16.3527   16.0439 0.7260          0.0000        1.0
1000000.0 1.0236 -16.4238   16.0455 0.7265          0.0000        1.0

Slide weights: train log-loss = 0.0


## Part D — Decision boundaries in 2-D (slides 18–19)

In [7]:
# Slide 18: w1 = w2 = 1, b = -3  ->  x1 + x2 = 3
for p in [(1, 1), (2, 1), (3, 0), (2, 2), (0, 4)]:
    z = p[0] + p[1] - 3
    print(f"point {p}: z = {z:2d}, g = {sig(z):.4f}, class {int(z >= 0)}")
print()
# Slide 19: features x1^2, x2^2, w1 = w2 = 1, b = -1  ->  circle x1^2 + x2^2 = 1
for p in [(0, 0), (0.5, 0.5), (1, 0), (1, 1), (0.6, 0.8)]:
    z = p[0] ** 2 + p[1] ** 2 - 1
    print(f"point {p}: z = {z:6.3f}, g = {sig(z):.4f}, class {int(z >= 0)}")

point (1, 1): z = -1, g = 0.2689, class 0
point (2, 1): z =  0, g = 0.5000, class 1
point (3, 0): z =  0, g = 0.5000, class 1
point (2, 2): z =  1, g = 0.7311, class 1
point (0, 4): z =  1, g = 0.7311, class 1

point (0, 0): z = -1.000, g = 0.2689, class 0
point (0.5, 0.5): z = -0.500, g = 0.3775, class 0
point (1, 0): z =  0.000, g = 0.5000, class 1
point (1, 1): z =  1.000, g = 0.7311, class 1
point (0.6, 0.8): z =  0.000, g = 0.5000, class 1


In [8]:
# Learning a circular boundary: plain LR fails, LR on squared features succeeds
rng = np.random.default_rng(0)
P = rng.uniform(-2, 2, size=(400, 2))
lab = (P[:, 0] ** 2 + P[:, 1] ** 2 >= 1.5).astype(int)
plain = LogisticRegression().fit(P, lab)
sq = LogisticRegression(C=100, max_iter=1000).fit(P ** 2, lab)
print("class balance:", lab.mean().round(3))
print("plain LR on (x1, x2)       train acc:", round(plain.score(P, lab), 4))
print("LR on (x1^2, x2^2)         train acc:", round(sq.score(P ** 2, lab), 4),
      "| weights", sq.coef_.round(3), "bias", sq.intercept_.round(3),
      "| implied radius^2 =", round(-sq.intercept_[0] / sq.coef_[0].mean(), 3))

class balance: 0.692
plain LR on (x1, x2)       train acc: 0.6925
LR on (x1^2, x2^2)         train acc: 0.9975 | weights [[18.87  18.689]] bias [-28.257] | implied radius^2 = 1.505


## Part E — Why not MSE (slides 20–25)

In [9]:
# one-parameter model p = g(w (x - 16)) on the word-count data, cost as a function of w
ws = np.linspace(-3, 3, 601)
def costs(w):
    z = w * (xw - 16)
    mse = np.mean((sig(z) - yw) ** 2) / 2
    ce = np.mean(yw * np.logaddexp(0, -z) + (1 - yw) * np.logaddexp(0, z))   # exact -log g, no clipping
    return mse, ce
mse, ce = np.array([costs(w) for w in ws]).T
d2 = lambda f: np.diff(f, 2)
print("MSE cost: some second difference < 0 (curves downward -> non-convex)?", (d2(mse) < -1e-12).any())
print("Log-loss: some second difference < 0?                                ", (d2(ce) < -1e-12).any())
print("MSE at w = -3, -1, 0, 1, 3:", np.round(mse[[0, 200, 300, 400, 600]], 4))
print("CE  at w = -3, -1, 0, 1, 3:", np.round(ce[[0, 200, 300, 400, 600]], 4))

MSE cost: some second difference < 0 (curves downward -> non-convex)? True
Log-loss: some second difference < 0?                                 False
MSE at w = -3, -1, 0, 1, 3: [0.5   0.5   0.125 0.    0.   ]
CE  at w = -3, -1, 0, 1, 3: [38.3077 12.7692  0.6931  0.      0.    ]


In [10]:
# per-sample loss table
for p in [0.99, 0.9, 0.7, 0.5, 0.3, 0.1, 0.01]:
    print(f"f = {p:4.2f}:  y=1 loss -log f = {-np.log(p):.4f}   y=0 loss -log(1-f) = {-np.log(1 - p):.4f}   squared error (y=1) = {(1 - p) ** 2:.4f}")

f = 0.99:  y=1 loss -log f = 0.0101   y=0 loss -log(1-f) = 4.6052   squared error (y=1) = 0.0001
f = 0.90:  y=1 loss -log f = 0.1054   y=0 loss -log(1-f) = 2.3026   squared error (y=1) = 0.0100
f = 0.70:  y=1 loss -log f = 0.3567   y=0 loss -log(1-f) = 1.2040   squared error (y=1) = 0.0900
f = 0.50:  y=1 loss -log f = 0.6931   y=0 loss -log(1-f) = 0.6931   squared error (y=1) = 0.2500
f = 0.30:  y=1 loss -log f = 1.2040   y=0 loss -log(1-f) = 0.3567   squared error (y=1) = 0.4900
f = 0.10:  y=1 loss -log f = 2.3026   y=0 loss -log(1-f) = 0.1054   squared error (y=1) = 0.8100
f = 0.01:  y=1 loss -log f = 4.6052   y=0 loss -log(1-f) = 0.0101   squared error (y=1) = 0.9801


In [11]:
# Gradient wrt z for one sample with y = 1 and f = g(z): MSE vs CE
for z in [-6, -3, 0, 3]:
    f = sig(z)
    g_mse = (f - 1) * f * (1 - f)     # d/dz of (f - y)^2 / 2
    g_ce = f - 1                       # d/dz of -log f
    print(f"z = {z:3d}  f = {f:.4f}  dMSE/dz = {g_mse:8.5f}  dCE/dz = {g_ce:8.5f}")

z =  -6  f = 0.0025  dMSE/dz = -0.00246  dCE/dz = -0.99753
z =  -3  f = 0.0474  dMSE/dz = -0.04303  dCE/dz = -0.95257
z =   0  f = 0.5000  dMSE/dz = -0.12500  dCE/dz = -0.50000
z =   3  f = 0.9526  dMSE/dz = -0.00214  dCE/dz = -0.04743


## Part F — Gradient descent from scratch

In [12]:
def fit_logreg_gd(X, y, alpha=0.1, iters=1000):
    X = np.asarray(X, float); y = np.asarray(y, float)
    w = np.zeros(X.shape[1]); b = 0.0; hist = []
    for _ in range(iters):
        f = sig(X @ w + b)
        dw = X.T @ (f - y) / len(y)
        db = np.mean(f - y)
        w -= alpha * dw; b -= alpha * db
        hist.append(log_loss(y, np.clip(f, 1e-15, 1 - 1e-15)))
    return w, b, hist

# Worked first iteration on the 5-point slide data, feature scaled to x/100
Xs = (X / 100)[:, None]
f0 = sig(Xs @ np.zeros(1) + 0)
print("initial f:", f0, "| residuals f - y:", f0 - y)
print("dw =", (Xs.T @ (f0 - y) / 5).round(5), " db =", np.mean(f0 - y).round(5))
w1, b1, _ = fit_logreg_gd(Xs, y, alpha=1.0, iters=1)
print("after one step (alpha = 1):  w =", w1.round(5), " b =", round(b1, 5))

initial f: [0.5 0.5 0.5 0.5 0.5] | residuals f - y: [ 0.5  0.5 -0.5 -0.5  0.5]
dw = [-0.097]  db = 0.1
after one step (alpha = 1):  w = [0.097]  b = -0.1


In [13]:
w_gd, b_gd, hist = fit_logreg_gd(Xs, y, alpha=1.0, iters=5000)
print("GD after 5000 steps: w =", w_gd.round(4), "b =", round(b_gd, 4), "| loss", round(hist[-1], 5))
print("boundary in word count:", round(-b_gd / w_gd[0] * 100, 2))
print("loss at steps 1, 10, 100, 1000, 5000:", [round(hist[i], 4) for i in [0, 9, 99, 999, 4999]])
sk = LogisticRegression(C=1e6, max_iter=10000).fit(Xs, y)
print("sklearn (C = 1e6):  w =", sk.coef_[0].round(4), "b =", sk.intercept_.round(4))
print("predictions:", (sig(Xs @ w_gd + b_gd)).round(4), "| P(x=20) =", round(sig(0.2 * w_gd[0] + b_gd), 4))

GD after 5000 steps: w = [32.7797] b = -10.9679 | loss 0.04524
boundary in word count: 33.46
loss at steps 1, 10, 100, 1000, 5000: [0.6931, 0.5687, 0.2991, 0.1272, 0.0452]
sklearn (C = 1e6):  w = [106.6876] b = [-35.9621]
predictions: [0.1074 0.     1.     0.8951 0.0017] | P(x=20) = 0.012


## Part G — Metrics (slides 26–40)

In [14]:
actual = np.array([0, 0, 1, 1, 0]); pred_lr = np.array([0, 0, 0, 1, 1]); pred_dt = np.array([0, 0, 1, 0, 0])
for name, p in [("LR", pred_lr), ("DT", pred_dt)]:
    tn, fp, fn, tp = confusion_matrix(actual, p).ravel()
    prec = precision_score(actual, p, zero_division=0); rec = recall_score(actual, p)
    print(f"{name}: TP={tp} TN={tn} FP={fp} FN={fn} | acc={accuracy_score(actual, p):.2f} "
          f"prec={prec:.2f} rec={rec:.2f} f1={f1_score(actual, p, zero_division=0):.4f}")
print("\nsklearn confusion_matrix layout (rows actual 0,1; cols predicted 0,1):")
print(confusion_matrix(actual, pred_lr))

LR: TP=1 TN=2 FP=1 FN=1 | acc=0.60 prec=0.50 rec=0.50 f1=0.5000
DT: TP=1 TN=3 FP=0 FN=1 | acc=0.80 prec=1.00 rec=0.50 f1=0.6667

sklearn confusion_matrix layout (rows actual 0,1; cols predicted 0,1):
[[2 1]
 [1 1]]


In [15]:
def metrics(tp, fp, fn, tn):
    tot = tp + fp + fn + tn
    prec = tp / (tp + fp) if tp + fp else float("nan")
    rec = tp / (tp + fn) if tp + fn else float("nan")
    f1 = 2 * tp / (2 * tp + fp + fn) if tp else 0.0
    spec = tn / (tn + fp)
    return {"acc": (tp + tn) / tot, "precision": prec, "recall": rec, "F1": f1,
            "specificity": spec, "balanced acc": (rec + spec) / 2 if rec == rec else float("nan")}

cases = {"terrorist: all 'not'   (slide 36)": (0, 0, 1, 9999),
         "terrorist: recall-max  (slide 38)": (1, 100, 0, 9899),
         "spam: precision-max    (slide 39)": (30, 1, 70, 9899)}
print(pd.DataFrame({k: metrics(*v) for k, v in cases.items()}).T.round(4))
print("\nF1 spam exact = 2*30/(2*30+1+70) = 60/131 =", round(60 / 131, 5))
print("F1 from rounded P, R: 2*0.967*0.30/1.267 =", round(2 * 0.967 * 0.30 / 1.267, 5))

                                      acc  precision  recall      F1  \
terrorist: all 'not'   (slide 36)  0.9999        NaN     0.0  0.0000   
terrorist: recall-max  (slide 38)  0.9900     0.0099     1.0  0.0196   
spam: precision-max    (slide 39)  0.9929     0.9677     0.3  0.4580   

                                   specificity  balanced acc  
terrorist: all 'not'   (slide 36)       1.0000        0.5000  
terrorist: recall-max  (slide 38)       0.9900        0.9950  
spam: precision-max    (slide 39)       0.9999        0.6499  

F1 spam exact = 2*30/(2*30+1+70) = 60/131 = 0.45802
F1 from rounded P, R: 2*0.967*0.30/1.267 = 0.45793


In [16]:
# F-beta: weight recall beta times as much as precision
def fbeta(p, r, beta):
    return (1 + beta ** 2) * p * r / (beta ** 2 * p + r)
p, r = 30 / 31, 0.30
for beta in [0.5, 1, 2]:
    print(f"spam  F{beta} = {fbeta(p, r, beta):.4f}")
p, r = 1 / 101, 1.0
for beta in [0.5, 1, 2]:
    print(f"terror F{beta} = {fbeta(p, r, beta):.4f}")

spam  F0.5 = 0.6696
spam  F1 = 0.4580
spam  F2 = 0.3480
terror F0.5 = 0.0123
terror F1 = 0.0196
terror F2 = 0.0476


## Part H — Thresholds, ROC and AUC

In [17]:
scores = np.array([0.95, 0.85, 0.80, 0.70, 0.55, 0.45, 0.40, 0.30, 0.20, 0.10])
truth  = np.array([1,    1,    0,    1,    1,    0,    0,    1,    0,    0])
for t in [0.9, 0.75, 0.5, 0.35, 0.15]:
    pr = (scores >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(truth, pr).ravel()
    print(f"threshold {t:4.2f}: TP={tp} FP={fp} FN={fn} TN={tn}  TPR={tp/(tp+fn):.2f}  FPR={fp/(fp+tn):.2f}  "
          f"precision={tp/(tp+fp) if tp+fp else float('nan'):.2f}")
fpr, tpr, thr = roc_curve(truth, scores)
print("\nROC points (FPR, TPR):", list(zip(fpr.round(2), tpr.round(2))))
print("AUC =", roc_auc_score(truth, scores))
pos, neg = scores[truth == 1], scores[truth == 0]
pairs = [(p > n) + 0.5 * (p == n) for p in pos for n in neg]
print(f"Pairs where the positive outranks the negative: {sum(pairs)} / {len(pairs)} = {np.mean(pairs)}")

threshold 0.90: TP=1 FP=0 FN=4 TN=5  TPR=0.20  FPR=0.00  precision=1.00
threshold 0.75: TP=2 FP=1 FN=3 TN=4  TPR=0.40  FPR=0.20  precision=0.67
threshold 0.50: TP=4 FP=1 FN=1 TN=4  TPR=0.80  FPR=0.20  precision=0.80
threshold 0.35: TP=4 FP=3 FN=1 TN=2  TPR=0.80  FPR=0.60  precision=0.57
threshold 0.15: TP=5 FP=4 FN=0 TN=1  TPR=1.00  FPR=0.80  precision=0.56

ROC points (FPR, TPR): [(np.float64(0.0), np.float64(0.0)), (np.float64(0.0), np.float64(0.2)), (np.float64(0.0), np.float64(0.4)), (np.float64(0.2), np.float64(0.4)), (np.float64(0.2), np.float64(0.8)), (np.float64(0.6), np.float64(0.8)), (np.float64(0.6), np.float64(1.0)), (np.float64(1.0), np.float64(1.0))]
AUC = 0.8
Pairs where the positive outranks the negative: 20.0 / 25 = 0.8


## Part I — Multi-class (iris)

In [18]:
iris = load_iris()
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(iris.data, iris.target, test_size=0.3, random_state=42, stratify=iris.target)
softmax = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xi_tr, yi_tr)
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier
ovr = make_pipeline(StandardScaler(), OneVsRestClassifier(LogisticRegression(max_iter=1000))).fit(Xi_tr, yi_tr)
ovo = make_pipeline(StandardScaler(), OneVsOneClassifier(LogisticRegression(max_iter=1000))).fit(Xi_tr, yi_tr)
for n, m in [("softmax (multinomial)", softmax), ("one-vs-rest", ovr), ("one-vs-one", ovo)]:
    print(f"{n:22s} test acc = {m.score(Xi_te, yi_te):.4f}")
print("\nnumber of binary models: OvR =", len(ovr[-1].estimators_), "| OvO =", len(ovo[-1].estimators_))
p0 = softmax.predict_proba(Xi_te[:1])[0]
print("softmax probabilities of first test flower:", p0.round(4), "sum =", p0.sum().round(6))

softmax (multinomial)  test acc = 0.9111
one-vs-rest            test acc = 0.8444
one-vs-one             test acc = 0.9111

number of binary models: OvR = 3 | OvO = 3
softmax probabilities of first test flower: [0.     0.1014 0.8985] sum = 1.0


In [19]:
pred = softmax.predict(Xi_te)
print(confusion_matrix(yi_te, pred))
print(classification_report(yi_te, pred, target_names=iris.target_names, digits=4))

[[15  0  0]
 [ 0 14  1]
 [ 0  3 12]]
              precision    recall  f1-score   support

      setosa     1.0000    1.0000    1.0000        15
  versicolor     0.8235    0.9333    0.8750        15
   virginica     0.9231    0.8000    0.8571        15

    accuracy                         0.9111        45
   macro avg     0.9155    0.9111    0.9107        45
weighted avg     0.9155    0.9111    0.9107        45



In [20]:
# micro vs macro vs weighted averaging on an imbalanced 3-class toy
yt = np.array([0]*80 + [1]*15 + [2]*5)
yp = np.array([0]*78 + [1]*2 + [1]*10 + [0]*5 + [0]*4 + [2]*1)
for avg in ["micro", "macro", "weighted"]:
    print(f"{avg:8s}: precision {precision_score(yt, yp, average=avg):.4f}  recall {recall_score(yt, yp, average=avg):.4f}  F1 {f1_score(yt, yp, average=avg):.4f}")
print("accuracy:", accuracy_score(yt, yp))
print(confusion_matrix(yt, yp))

micro   : precision 0.8900  recall 0.8900  F1 0.8900
macro   : precision 0.9100  recall 0.6139  F1 0.6694
weighted: precision 0.8922  recall 0.8900  F1 0.8751
accuracy: 0.89
[[78  2  0]
 [ 5 10  0]
 [ 4  0  1]]


## Part J — Model comparison demo (slide 40)
Six classifiers on the Wisconsin breast-cancer data (569 tumours, 30 features, 1 = benign), all inside a pipeline with scaling where it matters, 5-fold stratified cross-validation.

In [21]:
bc = load_breast_cancer()
Xb, yb = bc.data, bc.target
print(Xb.shape, "| benign share:", yb.mean().round(4))
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    "k-NN (k=5)": make_pipeline(StandardScaler(), KNeighborsClassifier(5)),
    "Gaussian Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM (RBF)": make_pipeline(StandardScaler(), SVC(random_state=42)),
}
cv = StratifiedKFold(5, shuffle=True, random_state=42)
rows = []
for name, m in models.items():
    acc = cross_val_score(m, Xb, yb, cv=cv, scoring="accuracy")
    f1m = cross_val_score(m, Xb, yb, cv=cv, scoring="f1_macro")
    auc = cross_val_score(m, Xb, yb, cv=cv, scoring="roc_auc")
    rows.append({"model": name, "acc mean": acc.mean(), "acc std": acc.std(), "F1 macro": f1m.mean(), "ROC AUC": auc.mean()})
comp = pd.DataFrame(rows).sort_values("acc mean", ascending=False).round(4)
print(comp.to_string(index=False))

(569, 30) | benign share: 0.6274


               model  acc mean  acc std  F1 macro  ROC AUC
           SVM (RBF)    0.9772   0.0163    0.9754   0.9945
 Logistic Regression    0.9737   0.0166    0.9714   0.9953
          k-NN (k=5)    0.9631   0.0179    0.9599   0.9849
       Random Forest    0.9543   0.0102    0.9509   0.9896
Gaussian Naive Bayes    0.9385   0.0235    0.9331   0.9887
       Decision Tree    0.9104   0.0279    0.9028   0.9000


In [22]:
# The malignant class is the one that matters: recall of malignant (label 0) on a held-out split
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.25, random_state=42, stratify=yb)
for name, m in models.items():
    p = m.fit(Xb_tr, yb_tr).predict(Xb_te)
    cm = confusion_matrix(yb_te, p)
    print(f"{name:22s} acc {accuracy_score(yb_te, p):.4f} | malignant recall {recall_score(yb_te, p, pos_label=0):.4f} "
          f"| missed malignant {cm[0, 1]} of {cm[0].sum()}")

Logistic Regression    acc 0.9860 | malignant recall 0.9811 | missed malignant 1 of 53
k-NN (k=5)             acc 0.9790 | malignant recall 0.9434 | missed malignant 3 of 53
Gaussian Naive Bayes   acc 0.9371 | malignant recall 0.8679 | missed malignant 7 of 53
Decision Tree          acc 0.9231 | malignant recall 0.9245 | missed malignant 4 of 53


Random Forest          acc 0.9580 | malignant recall 0.9245 | missed malignant 4 of 53
SVM (RBF)              acc 0.9790 | malignant recall 0.9811 | missed malignant 1 of 53


In [23]:
# Unscaled k-NN and LR: scaling matters for distance / gradient models, not for trees
for name, m in {"k-NN unscaled": KNeighborsClassifier(5), "k-NN scaled": models["k-NN (k=5)"],
                "Tree unscaled": DecisionTreeClassifier(random_state=42),
                "Tree scaled": make_pipeline(StandardScaler(), DecisionTreeClassifier(random_state=42))}.items():
    print(f"{name:14s} CV acc {cross_val_score(m, Xb, yb, cv=cv).mean():.4f}")

k-NN unscaled  CV acc 0.9350
k-NN scaled    CV acc 0.9631
Tree unscaled  CV acc 0.9104


Tree scaled    CV acc 0.9104


In [24]:
# Generative vs discriminative with little data (Ng & Jordan 2001 effect): mean test acc over 30 random small training sets
res = {}
for n in [10, 20, 40, 100, 300]:
    accs = {"GaussianNB": [], "LogReg": []}
    for seed in range(30):
        Xtr, Xte, ytr, yte = train_test_split(Xb, yb, train_size=n, random_state=seed, stratify=yb)
        accs["GaussianNB"].append(GaussianNB().fit(Xtr, ytr).score(Xte, yte))
        accs["LogReg"].append(make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(Xtr, ytr).score(Xte, yte))
    res[n] = {k: np.mean(v) for k, v in accs.items()}
print(pd.DataFrame(res).T.round(4))

     GaussianNB  LogReg
10       0.9089  0.9044
20       0.9271  0.9276
40       0.9366  0.9502
100      0.9360  0.9652
300      0.9354  0.9743


## Part K — Class imbalance

In [25]:
Xi, yi = make_classification(n_samples=20000, n_features=10, n_informative=4, weights=[0.98, 0.02],
                             flip_y=0.005, class_sep=1.0, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(Xi, yi, test_size=0.3, random_state=0, stratify=yi)
print("positive share:", yi.mean().round(4))
dummy_acc = (yte == 0).mean()
print("always-0 accuracy:", round(dummy_acc, 4))
base = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
bal = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr, ytr)
prob = base.predict_proba(Xte)[:, 1]
def rep(name, p):
    print(f"{name:28s} acc {accuracy_score(yte, p):.4f}  precision {precision_score(yte, p, zero_division=0):.4f}  "
          f"recall {recall_score(yte, p):.4f}  F1 {f1_score(yte, p):.4f}  bal-acc {balanced_accuracy_score(yte, p):.4f}")
rep("LR, threshold 0.5", base.predict(Xte))
rep("LR, threshold 0.2", (prob >= 0.2).astype(int))
rep("LR class_weight='balanced'", bal.predict(Xte))
print("ROC AUC (threshold-free):", round(roc_auc_score(yte, prob), 4))

positive share: 0.0221
always-0 accuracy: 0.9778
LR, threshold 0.5            acc 0.9775  precision 0.0000  recall 0.0000  F1 0.0000  bal-acc 0.4998
LR, threshold 0.2            acc 0.9812  precision 0.6613  recall 0.3083  F1 0.4205  bal-acc 0.6523
LR class_weight='balanced'   acc 0.7002  precision 0.0522  recall 0.7293  F1 0.0973  bal-acc 0.7144
ROC AUC (threshold-free): 0.7443


## Part L — Practice-problem solutions

In [26]:
# P: metrics from confusion matrices
for name, (tp, fp, fn, tn) in {"P3 (40, 10, 20, 130)": (40, 10, 20, 130),
                                "P4 disease screen (90, 495, 10, 9405)": (90, 495, 10, 9405)}.items():
    print(name, {k: round(v, 4) for k, v in metrics(tp, fp, fn, tn).items()})

P3 (40, 10, 20, 130) {'acc': 0.85, 'precision': 0.8, 'recall': 0.6667, 'F1': 0.7273, 'specificity': 0.9286, 'balanced acc': 0.7976}
P4 disease screen (90, 495, 10, 9405) {'acc': 0.9495, 'precision': 0.1538, 'recall': 0.9, 'F1': 0.2628, 'specificity': 0.95, 'balanced acc': 0.925}


In [27]:
# P: sigmoid / boundary hand checks
print("P1: g(2) =", round(sig(2), 4), "| g(-0.5) =", round(sig(-0.5), 4), "| z for p = 0.9:", round(np.log(9), 4))
w1_, w2_, b_ = 2, -1, -4
for pt in [(3, 1), (1, 1), (2, 0)]:
    z = w1_ * pt[0] + w2_ * pt[1] + b_
    print(f"P2: point {pt}: z = {z}, p = {sig(z):.4f}, class {int(z >= 0)}")
print("P5 log-loss of predictions [0.9, 0.2, 0.6] for y = [1, 0, 0]:",
      round(log_loss([1, 0, 0], [0.9, 0.2, 0.6]), 4), "| terms",
      np.round([-np.log(0.9), -np.log(0.8), -np.log(0.4)], 4))

P1: g(2) = 0.8808 | g(-0.5) = 0.3775 | z for p = 0.9: 2.1972
P2: point (3, 1): z = 1, p = 0.7311, class 1
P2: point (1, 1): z = -3, p = 0.0474, class 0
P2: point (2, 0): z = 0, p = 0.5000, class 1
P5 log-loss of predictions [0.9, 0.2, 0.6] for y = [1, 0, 0]: 0.4149 | terms [0.1054 0.2231 0.9163]


In [28]:
# P: AUC by pair counting
s = np.array([0.9, 0.7, 0.6, 0.4, 0.3]); t = np.array([1, 0, 1, 0, 0])
print("P8 AUC =", roc_auc_score(t, s))

P8 AUC = 0.8333333333333334
